# CPU smoke test: distilgpt2 + LoRA
Use this notebook to run a quick CPU-only fine-tune on a small sample.

In [1]:
import os
from pathlib import Path
import pandas as pd
import json

ROOT = Path('.')
DATA_CSV = ROOT / 'finance moule' / 'InternationalDeclarations.csv'
FT_DATA_DIR = ROOT / 'ft_data'
FT_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = ROOT / 'outputs' / 'ft-run'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_CSV:', DATA_CSV)
print('FT_DATA_DIR:', FT_DATA_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)

DATA_CSV: finance moule/InternationalDeclarations.csv
FT_DATA_DIR: ft_data
OUTPUT_DIR: outputs/ft-run


In [2]:
# Create a small sample JSONL for a quick smoke test
df = pd.read_csv(DATA_CSV)
print(f'Loaded {len(df)} rows from CSV')

SAMPLE_SIZE = 500
if len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=42)
    print(f'Sampled to {len(df)} rows')

def create_training_pairs(df):
    pairs = []
    for _, row in df.iterrows():
        concept = row.get('concept:name', 'UNKNOWN')
        permit_id = row.get('case:id', row.get('id', ''))
        org_role = row.get('org:role', '')
        amount = row.get('case:Amount', '')

        instruction = f'Describe the following financial event: {concept} for permit {permit_id}'
        response = f'Event: {concept}. Role: {org_role}. Amount: {amount}. Status: processed.'

        pairs.append({
            'instruction': instruction,
            'input': '',
            'output': response
        })
    return pairs

pairs = create_training_pairs(df)
train_jsonl = FT_DATA_DIR / 'train.jsonl'
with open(train_jsonl, 'w', encoding='utf8') as f:
    for pair in pairs:
        f.write(json.dumps(pair, ensure_ascii=False) + '\n')

print(f'Created {len(pairs)} instruction-response pairs')
print(f'Saved to: {train_jsonl}')

FileNotFoundError: [Errno 2] No such file or directory: 'finance moule/InternationalDeclarations.csv'

In [ ]:
# CPU training cell: distilgpt2 + LoRA (small smoke test)
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, default_data_collator
import torch
from peft import LoraConfig, get_peft_model

MODEL_PATH = 'distilgpt2'

print('Loading dataset...')
ds = load_dataset('json', data_files=str(train_jsonl))
print(f'Dataset size: {len(ds[
]) } examples')

def make_text(example):
    inst = example.get('instruction', '')
    inp = example.get('input', '')
    out = example.get('output', '')
    if inp:
        text = f'### Instruction:\n{inst}\n\n### Input:\n{inp}\n\n### Response:\n{out}\n'
    else:
        text = f'### Instruction:\n{inst}\n\n### Response:\n{out}\n'
    return {'text': text}

ds = ds.map(make_text, remove_columns=ds['train'].column_names)

print('Loading tokenizer and model (CPU)...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, device_map=None, torch_dtype=torch.float32, low_cpu_mem_usage=True)

# Attach LoRA adapters (r small for CPU run)
lora_config = LoraConfig(
    r=4,
    lora_alpha=16,
    target_modules=['c_attn'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

def tokenize_fn(batch):
    toks = tokenizer(batch['text'], truncation=True, max_length=256, padding='max_length')
    toks['labels'] = toks['input_ids'].copy()
    return toks

tok_ds = ds.map(tokenize_fn, batched=True)
tok_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

data_collator = default_data_collator

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    gradient_checkpointing=False,
    num_train_epochs=1,
    learning_rate=1e-4,
    fp16=False,
    logging_steps=50,
    report_to=[],
    save_total_limit=1,
    remove_unused_columns=False,
    dataloader_num_workers=0
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tok_ds['train'],
    data_collator=data_collator,
)

trainer.train()
print('Training complete!')

final_model_path = OUTPUT_DIR / 'final_cpu'
trainer.save_model(str(final_model_path))
print(f'LoRA adapters saved to: {final_model_path}')